# SPORES Inventory

Create a small, reviewable manifest for the local SPORES NetCDF files and check whether the expected filename families are complete.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "model_files").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

SPORES_DIR = ROOT / "results" / "spores"
SPORES_DIR

In [ ]:
from calliope_nl_analysis.spores import build_spores_manifest, validate_spore_inventory
from calliope_nl_analysis.distribution import distribution_summary

inventory = validate_spore_inventory(SPORES_DIR)
inventory

In [ ]:
manifest = build_spores_manifest(SPORES_DIR, include_sha256=False, root=ROOT)
manifest.head()

In [ ]:
family_summary = (
    manifest.groupby("family")
    .agg(files=("name", "count"), size_gib=("size_bytes", lambda values: values.sum() / 1024**3))
    .round(3)
)
family_summary

In [ ]:
MANIFEST_PATH = ROOT / "metadata" / "spores_manifest.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(MANIFEST_PATH, index=False)
MANIFEST_PATH

## Distribution Check

The helper below estimates the per-family archive sizes. For GitHub Releases, family ZIP files are usually a better fit than one very large ZIP because users can download only the families they need.

In [ ]:
summary = distribution_summary(SPORES_DIR)
summary["family_sizes"]

In [ ]:
print("Manifest only:")
print("python scripts/prepare_spores_distribution.py --output-dir metadata --archives none")

print("\nManifest plus one ZIP per family:")
print("python scripts/prepare_spores_distribution.py --output-dir dist/spores --archives family")